# Lab 3 — Evaluate base vs fine-tuned OpenVLA

A VLA outputs numbers, not text, so the serverless lab's LLM-as-judge and text scorers don't apply. We
score **closed-form** instead: for each validation frame, greedy-decode the 7 action tokens and compare
them with the label:

* `l1_mean`: mean absolute error over all 7 axes (normalized units, [-1, 1])
* `l1_x`, `l1_y`: the two axes that carry signal in this pseudo-labelled data
* `bin_accuracy`: fraction of axes whose prediction falls in the same 1/256 bin as the label
* `mean_action_baseline`: always predict the mean training action. A model that doesn't beat it has
  learned nothing beyond the label prior.

The comparison runs as a single-GPU SageMaker job (`scripts/evaluate_openvla.py`), so the notebook
kernel needs no GPU. The job evaluates the base model first, then the merged model.

In [ ]:
import boto3
from sagemaker.core.helper.session_helper import Session, get_execution_role

import config

sess = Session()
try:
    role = get_execution_role()
except ValueError:
    role = boto3.client("iam").get_role(RoleName="sagemaker_execution_role")["Role"]["Arn"]

region = sess.boto_region_name
bucket_name = sess.default_bucket()
s3_root = config.s3_root(bucket_name, sess.default_bucket_prefix)
dataset_s3_uri = f"{s3_root}/datasets/{config.DATASET_PREFIX}/"
s3 = boto3.client("s3", region_name=region)
sm_client = boto3.client("sagemaker", region_name=region)

print(f"role:     {role}")
print(f"region:   {region}")
print(f"run mode: {config.RUN_MODE}, capacity: {config.CAPACITY} -> {', '.join(config.TRAIN['instance_types'])}")
print(f"s3 root:  {s3_root}")

In [ ]:
def latest_package(status=None):
    kwargs = {"ModelPackageGroupName": config.MODEL_PACKAGE_GROUP, "SortBy": "CreationTime", "SortOrder": "Descending"}
    if status:
        kwargs["ModelApprovalStatus"] = status
    pkgs = sm_client.list_model_packages(**kwargs)["ModelPackageSummaryList"]
    if not pkgs:
        raise RuntimeError(f"no {status or ''} model package in {config.MODEL_PACKAGE_GROUP}; run the previous notebook")
    return sm_client.describe_model_package(ModelPackageName=pkgs[0]["ModelPackageArn"])

In [ ]:
# MLflow tracking (optional; metrics also appear in the job log either way)
#   MLFLOW_APP_ARN = "arn:aws:sagemaker:..."  -> use this MLflow App
#   MLFLOW_APP_ARN = ""                       -> reuse the workshop's app, or create one if the account has none
#                                                (creating takes several minutes, once)
#   USE_MLFLOW = False                        -> skip MLflow
USE_MLFLOW = config.USE_MLFLOW
MLFLOW_APP_ARN = config.MLFLOW_APP_ARN

from config import resolve_mlflow_app

mlflow_app_arn = resolve_mlflow_app(sm_client, session=sess, arn=MLFLOW_APP_ARN) if USE_MLFLOW else None
mlflow_env = {"MLFLOW_TRACKING_URI": mlflow_app_arn, "MLFLOW_EXPERIMENT_NAME": config.MLFLOW_EXPERIMENT} if mlflow_app_arn else {}
print(f"MLflow: {mlflow_app_arn or 'off'}")

package = latest_package()
merged_s3_uri = package["CustomerMetadataProperties"]["merged_s3_uri"]
print(f"package: {package['ModelPackageArn']} ({package['ModelApprovalStatus']})\nmodel:   {merged_s3_uri}")

In [ ]:
import time

from job_utils import latest_job, print_phases, wait_for_job

In [ ]:
from sagemaker.core.training.configs import InputData, OutputDataConfig, SourceCode
from sagemaker.train import ModelTrainer

from job_utils import capacity_config, with_quota

eval_name = f"{config.PROJECT}-eval-{time.strftime('%Y%m%d-%H%M%S')}"
# SageMaker rejects the whole list if any type in it has no quota, so keep only those with quota
instance_types = with_quota(config.EVAL_INSTANCE_TYPES, region, usage="spot training job usage" if config.CAPACITY == "spot" else "training job usage")
compute, stopping_condition = capacity_config(
    instance_types, config.CAPACITY, volume_size_in_gb=100,
    max_runtime_s=2 * 3600, spot_max_wait_h=config.SPOT_MAX_WAIT_H)
evaluator = ModelTrainer(
    training_image=config.TRAINING_IMAGE.format(region=region),
    source_code=SourceCode(source_dir="scripts", requirements="requirements.txt", entry_script="run_evaluation.sh"),
    compute=compute,
    stopping_condition=stopping_condition,
    output_data_config=OutputDataConfig(s3_output_path=f"{s3_root}/eval-output", compression_type="NONE"),
    environment=mlflow_env,
    base_job_name=eval_name[:48],
    role=role,
    sagemaker_session=sess,
)
evaluator.train(input_data_config=[InputData(channel_name="data", data_source=dataset_s3_uri),
                                   InputData(channel_name="model", data_source=merged_s3_uri)], wait=False)
eval_job_name = evaluator._latest_training_job.training_job_name
print(f"submitted: {eval_job_name}")

### Wait for the evaluation job, or reconnect to it

Same as in notebook 2: stopping this cell does not stop the job. After a kernel restart, run the setup cells
and the `job_utils` import, then this cell; it finds the most recent evaluation job.

In [ ]:
eval_job_name = globals().get("eval_job_name") or latest_job(f"{config.PROJECT}-eval")
eval_job = wait_for_job(eval_job_name)
print_phases(eval_job)

In [ ]:
import json

import pandas as pd

results_uri = eval_job["ModelArtifacts"]["S3ModelArtifacts"].rstrip("/") + "/eval_results.json"
bucket, key = results_uri[5:].split("/", 1)
results = json.loads(s3.get_object(Bucket=bucket, Key=key)["Body"].read())

cols = ["mean_action_baseline", "base", "finetuned"]
table = pd.DataFrame({c: results[c] for c in cols})
table["finetuned vs base"] = (table["finetuned"] - table["base"]) / table["base"].abs()
print(f"{results['num_samples']} validation samples ({results_uri})")
table.style.format("{:.3f}").format("{:+.0%}", subset=["finetuned vs base"])

## Approve the package

The gate compares `l1_xy`, the L1 on the two axes that carry signal (the 7-axis `l1_mean` is dominated by the
5 constant axes). The package is approved if the fine-tuned model beats the base model.

The mean-action baseline is printed as context, not used as a gate. With this lab's pseudo-labels (mean optical
flow, one fixed instruction) a 10-epoch run on the 300-episode sample beats the base model clearly but lands
close to the label prior: the model learns the label distribution more than per-image motion. That is a
property of the synthetic labels, not of the pipeline; with real robot actions, expect the fine-tuned model
to beat the baseline as well.

In [ ]:
l1_xy = {k: (results[k]["l1_x"] + results[k]["l1_y"]) / 2 for k in ("finetuned", "base", "mean_action_baseline")}
ft, base, prior = l1_xy["finetuned"], l1_xy["base"], l1_xy["mean_action_baseline"]
approve = ft < base
summary = f"l1_xy finetuned={ft:.3f} base={base:.3f} (mean-action baseline {prior:.3f})"
sm_client.update_model_package(
    ModelPackageArn=package["ModelPackageArn"],
    ModelApprovalStatus="Approved" if approve else "Rejected",
    ApprovalDescription=summary,
)
print("Approved" if approve else "Rejected", f"({summary})")
print("beats the label prior" if ft < prior else "does not beat the label prior (expected with pseudo-labels)")

Continue with **4-deployment.ipynb**.